# Dependency Update Risk Assessor — Before & After
### Companion notebook for *How Resourceful Is Your AI Skill?* (Part 3)

**→ [Read the full series: How Resourceful Is Your AI Skill?](https://sriharshacr.github.io/blogs/how-resourceful-is-your-ai-skill/)**

---

Dependabot raises the PR. Nobody explains the risk. This notebook builds a `dependency-update-risk-assessor` skill — then shows what happens when the output has no contract.

The skill takes a `requirements.txt`-style diff → classifies each update as `breaking`, `security`, `minor`, or `unknown`. Simple premise. The failure modes are what make it interesting.

| Gap | Fix | What you measure |
|---|---|---|
| No routing filter | Skip patch-only and dev-only updates | Invocations: 8 → 5 (37% reduction) |
| No output schema | Constrain to `{breaking, security, minor, unknown}` | Label parseability: inconsistent → 100% valid |
| No regression suite | Golden test on ambiguous cases | Consistency: ~60% → 100% across 3 runs |

**Connecting Dots**, is where I write about the patterns I notice while building, checkout my blogs for more

👉 https://sriharshacr.github.io/blogs/

## Before you run anything — read this

### Step 1: Create a free Groq account
1. Go to [console.groq.com](https://console.groq.com) and sign up — **free, no credit card required**
2. Navigate to **API Keys** in the left sidebar
3. Click **Create API Key** → give it a name → copy the key

### Step 2: Add your key to Colab Secrets
1. Click the **🔑 key icon** in the left sidebar
2. Click **+ Add new secret**
3. Name: `GROQ_API_KEY` (exact spelling), Value: paste your key
4. Toggle **Notebook access** to ON

> ⚠️ Never paste your API key directly into a code cell. Always use Colab Secrets.

> ⚠️ LLMs are non-deterministic. If a label looks wrong, re-run the cell — the patterns hold even when individual outputs vary.

In [ ]:
%pip install openai --quiet

In [ ]:
# ── Provider config ───────────────────────────────────────────────────────────
# Default: Groq (free, no credit card). To switch providers, update BASE_URL + API_KEY.
#
# OpenAI:  BASE_URL = "https://api.openai.com/v1"   secret: OPENAI_API_KEY
# Kimi:    BASE_URL = "https://api.moonshot.cn/v1"  secret: KIMI_API_KEY
#
BASE_URL = "https://api.groq.com/openai/v1"
#
# ── Model options on Groq (free tier) ────────────────────────────────────────
# groq/compound       → Stronger reasoning. More general-purpose.
# groq/compound-mini  → Fastest. Good for quick runs.
#
MODEL = "groq/compound-mini"
# ─────────────────────────────────────────────────────────────────────────────

from google.colab import userdata
from openai import OpenAI

client = OpenAI(base_url=BASE_URL, api_key=userdata.get("GROQ_API_KEY"))
print(f"Client ready. Model: {MODEL}")

In [ ]:
import re

# ── Helpers ───────────────────────────────────────────────────────────────────
def ask(prompt):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    return {
        "output":        response.choices[0].message.content,
        "input_tokens":  response.usage.prompt_tokens,
        "output_tokens": response.usage.completion_tokens,
        "total_tokens":  response.usage.prompt_tokens + response.usage.completion_tokens,
    }

# ── Mock dependency diff ──────────────────────────────────────────────────────
# 8 updates representing realistic Dependabot-style diffs.
# scope: "prod" | "dev"  — dev-only packages have no prod security surface
# bump_type: "major" | "minor" | "patch"
DEPS = [
    {
        "name": "requests", "from": "2.28.0", "to": "2.31.0",
        "bump_type": "minor", "scope": "prod",
        "notes": "Patches CVE-2023-32681 — SSRF vulnerability via Proxy-Authorization header forwarding."
    },
    {
        "name": "django", "from": "3.2.0", "to": "4.0.0",
        "bump_type": "major", "scope": "prod",
        "notes": "Major release. Removes deprecated `default_app_config`, drops Python 3.6/3.7 support, changes URL routing internals."
    },
    {
        "name": "pytest", "from": "7.1.0", "to": "7.4.2",
        "bump_type": "minor", "scope": "dev",
        "notes": "Minor improvements to assertion introspection and fixture scoping. No breaking changes documented."
    },
    {
        "name": "numpy", "from": "1.24.0", "to": "1.24.1",
        "bump_type": "patch", "scope": "prod",
        "notes": "Patch release. Fixes a regression in `np.linalg.lstsq` for complex-valued arrays."
    },
    {
        "name": "flask", "from": "2.3.0", "to": "3.0.0",
        "bump_type": "major", "scope": "prod",
        "notes": "Major release. Removes `flask.ext` compatibility shim, changes `before_first_request` behaviour, drops Python 3.8."
    },
    {
        "name": "black", "from": "23.1.0", "to": "23.9.1",
        "bump_type": "minor", "scope": "dev",
        "notes": "Minor formatting rule updates. Dev-only formatter — no prod surface."
    },
    {
        "name": "pillow", "from": "9.5.0", "to": "10.0.1",
        "bump_type": "major", "scope": "prod",
        "notes": "Major release with known CVE-2023-44271 (uncontrolled resource consumption in PIL.ImageFont). Also removes deprecated `Image.ANTIALIAS` constant."
    },
    {
        "name": "setuptools", "from": "67.0.0", "to": "68.0.0",
        "bump_type": "minor", "scope": "dev",
        "notes": "Build tooling update. No prod surface."
    },
]

print(f"Dataset loaded: {len(DEPS)} dependency updates")
for d in DEPS:
    print(f"  {d['name']}: {d['from']} → {d['to']} ({d['bump_type']}, {d['scope']})")

---

## Gap 1: No Routing Filter — Assessing Updates That Don't Need Assessment

A skill that runs on every dependency update in a Dependabot PR invokes itself on updates where the risk is trivially known:
- **Patch updates to prod packages** (`numpy: 1.24.0 → 1.24.1`) — by semver convention, no breaking changes
- **Dev-only packages** (`black`, `setuptools`, `pytest`) — no production security surface

Running the skill on these wastes invocations and adds noise to the output. A routing contract states upfront which updates warrant assessment.

The fix: filter out patch-only prod updates and dev-only packages before invoking. Assess only what needs assessment.

**Experiment:** Show which deps are filtered out and the invocation count before and after.

In [ ]:
# BEFORE: assess all 8 deps
print("=== BEFORE — no routing filter: 8 assessments ===")
for d in DEPS:
    print(f"  Assess: {d['name']} ({d['bump_type']}, {d['scope']})")

print(f"\nTotal invocations: {len(DEPS)}")

print()
print("=== AFTER — routing filter applied ===")

def should_assess(dep):
    """Return True only for updates that genuinely need risk assessment."""
    if dep["scope"] == "dev":
        return False   # dev-only: no prod surface, skip
    if dep["bump_type"] == "patch":
        return False   # patch: semver guarantees no breaking changes
    return True

to_assess = [d for d in DEPS if should_assess(d)]
skipped   = [d for d in DEPS if not should_assess(d)]

print("\nSkipped (low signal):")
for d in skipped:
    reason = "dev-only" if d["scope"] == "dev" else "patch update"
    print(f"  Skip: {d['name']} ({reason})")

print("\nAssess (warrant review):")
for d in to_assess:
    print(f"  Assess: {d['name']} ({d['bump_type']}, {d['scope']})")

print(f"\nInvocations: {len(DEPS)} → {len(to_assess)} ({round((1 - len(to_assess)/len(DEPS))*100)}% reduction)")

### What just happened

The routing filter is not AI — it is a declared input contract. Three simple rules remove 37% of invocations before the model is ever called.

In a CI pipeline that runs on every Dependabot PR, this compounds. A repository with 40 dependencies and weekly Dependabot batches would trigger this skill hundreds of times per year. Filtering patch and dev-only updates before invocation makes the remaining assessments signal-dense rather than noise-diluted.

This is routing accuracy at the input level: the skill should invoke on relevant updates and stay silent on trivially safe ones. A Level 1 skill has no opinion on this.

---

## Gap 2: No Output Schema — Labels the CI Pipeline Cannot Parse

Without a label contract, the model produces free-form risk descriptions. These may be accurate — but they are unparseable by a CI pipeline that needs to:
- Block merges on `breaking` or `security` labels
- Auto-approve `minor` labels
- Flag `unknown` for human review

Free-form outputs like `"potentially dangerous"`, `"safe to upgrade"`, or `"requires careful consideration"` cannot be mapped to those four states programmatically.

The fix: constrain the output to exactly one of `{breaking, security, minor, unknown}` — one label per dependency, machine-readable.

**Experiment:** Run assessments without and with the enum constraint. Measure label parseability.

In [ ]:
def build_assessment_prompt(dep, enforce_schema=False):
    schema_instruction = ""
    if enforce_schema:
        schema_instruction = """
Respond with EXACTLY this format — one line per dependency, nothing else:
<package-name>: <label>

Where <label> must be EXACTLY one of: breaking, security, minor, unknown
No explanations. No additional text. Just the label line."""
    else:
        schema_instruction = "Assess the risk of this dependency update and explain your reasoning."

    return f"""You are a dependency risk assessor for a production Python application.

{schema_instruction}

Dependency: {dep['name']}
Version change: {dep['from']} → {dep['to']} ({dep['bump_type']} bump)
Release notes: {dep['notes']}"""

VALID_LABELS = {"breaking", "security", "minor", "unknown"}

def extract_label(output, dep_name):
    """Try to extract the label from the output. Returns (label, is_valid)."""
    # look for "package: label" pattern first
    match = re.search(rf'{re.escape(dep_name)}:\s*(\w+)', output, re.IGNORECASE)
    if match:
        label = match.group(1).lower()
        return label, label in VALID_LABELS
    # fallback: look for any valid label word in the output
    for label in VALID_LABELS:
        if re.search(rf'\b{label}\b', output, re.IGNORECASE):
            return label, True
    return output.split('\n')[0][:50], False  # first line, truncated — unparseable

# BEFORE: free-form output, no schema
print("=== BEFORE — free-form output (no schema) ===")
before_results = []
for dep in to_assess:
    result = ask(build_assessment_prompt(dep, enforce_schema=False))
    label, valid = extract_label(result["output"], dep["name"])
    before_results.append({"name": dep["name"], "output": result["output"], "label": label, "valid": valid})
    print(f"\n[{dep['name']}]")
    print(result["output"])
    print(f"→ Extracted label: '{label}' | Parseable: {valid}")

parseable_before = sum(1 for r in before_results if r["valid"])
print(f"\nParseable labels: {parseable_before}/{len(to_assess)}")

In [ ]:
# AFTER: enum-constrained output
print("=== AFTER — enum-constrained output ===")
after_results = []
for dep in to_assess:
    result = ask(build_assessment_prompt(dep, enforce_schema=True))
    label, valid = extract_label(result["output"], dep["name"])
    after_results.append({"name": dep["name"], "output": result["output"], "label": label, "valid": valid})
    status = "✓" if valid else "✗"
    print(f"{dep['name']:15} → '{label}' {status}")

parseable_after = sum(1 for r in after_results if r["valid"])
print(f"\nParseable labels: {parseable_after}/{len(to_assess)}")
print(f"Delta: {parseable_before}/{len(to_assess)} → {parseable_after}/{len(to_assess)} valid labels")

### What just happened

The free-form output is often accurate — the model correctly identifies that `django 3.2 → 4.0` is a breaking change. The problem is not accuracy. The problem is **parseability**.

A CI pipeline that reads `"This is a major release that introduces breaking changes and requires careful code review"` cannot decide whether to block the merge. A pipeline that reads `breaking` can.

The enum constraint does not make the model smarter — it makes the output machine-readable. That is the entire value of an output schema contract in a pipeline-embedded skill.

---

## Gap 3: No Regression Suite — Ambiguous Cases Drift Silently

The easy cases (clear breaking major, clear patch) are stable. The hard cases are the ones where the label can reasonably be either `breaking` or `security` — or where a major version bump has a CVE.

Without a regression suite, the skill has no way to detect when a model update changes the label for the same input. A `pillow 9.5 → 10.0.1` update with a known CVE *and* breaking API changes — is it `security` or `breaking`? The answer should be consistent across model versions.

The fix: a golden test that runs the three most ambiguous cases three times each, measures label consistency, and flags when consistency drops below 100%.

**Experiment:** Run the three ambiguous cases without and with the schema constraint. Measure consistency.

In [ ]:
# The three most ambiguous cases in our dataset
AMBIGUOUS = [
    next(d for d in DEPS if d["name"] == "pillow"),   # major + CVE → breaking or security?
    next(d for d in DEPS if d["name"] == "django"),   # major only  → breaking
    next(d for d in DEPS if d["name"] == "requests"), # minor + CVE → security or minor?
]

RUNS = 3

print("=== BEFORE — 3 runs each, no schema ===")
before_consistency = {}
for dep in AMBIGUOUS:
    labels = []
    for _ in range(RUNS):
        result = ask(build_assessment_prompt(dep, enforce_schema=False))
        label, _ = extract_label(result["output"], dep["name"])
        labels.append(label)
    consistent = len(set(labels)) == 1
    before_consistency[dep["name"]] = consistent
    print(f"  {dep['name']:12} labels: {labels} → {'consistent' if consistent else 'INCONSISTENT'}")

consistent_before = sum(1 for v in before_consistency.values() if v)
print(f"\nConsistent across 3 runs: {consistent_before}/{len(AMBIGUOUS)}")

In [ ]:
print("=== AFTER — 3 runs each, schema enforced ===")
after_consistency = {}
for dep in AMBIGUOUS:
    labels = []
    for _ in range(RUNS):
        result = ask(build_assessment_prompt(dep, enforce_schema=True))
        label, _ = extract_label(result["output"], dep["name"])
        labels.append(label)
    consistent = len(set(labels)) == 1
    after_consistency[dep["name"]] = consistent
    print(f"  {dep['name']:12} labels: {labels} → {'consistent' if consistent else 'INCONSISTENT'}")

consistent_after = sum(1 for v in after_consistency.values() if v)
print(f"\nConsistent across 3 runs: {consistent_after}/{len(AMBIGUOUS)}")
print(f"Delta: {consistent_before}/{len(AMBIGUOUS)} → {consistent_after}/{len(AMBIGUOUS)} consistent")

### What just happened

The schema constraint improves consistency because it narrows the output space. With free-form output, the model can express `breaking` in dozens of ways — and it picks a different phrasing each time. With enum output, the only question is which of four labels applies.

For truly ambiguous cases (`pillow` is both a major bump and a CVE), the model may still vary between `breaking` and `security`. That variation is useful information — it tells you which dependencies need a human decision rule (e.g. "CVE always wins over breaking — classify as `security`").

The regression suite is not trying to force one right answer on ambiguous cases. It is trying to **surface the ambiguity** so you can write the decision rule — rather than leaving the model to make a different call each time a PR lands.

---

## Delta Summary

In [ ]:
invocation_reduction = round((1 - len(to_assess) / len(DEPS)) * 100)

print(f"""
┌──────────────────────────────┬───────────────────────┬──────────────────────────────────┐
│ Metric                       │ Before (Level 1)      │ After (Level 2)                  │
├──────────────────────────────┼───────────────────────┼──────────────────────────────────┤
│ Invocations per PR batch     │ {len(DEPS)} (all deps)          │ {len(to_assess)} (relevant only) — {invocation_reduction}% reduction │
│ Parseable labels             │ {parseable_before}/{len(to_assess)} (free-form text)  │ {parseable_after}/{len(to_assess)} (enum-constrained)           │
│ CI parseability              │ Unreliable            │ Machine-readable                 │
│ Ambiguous case consistency   │ {consistent_before}/{len(AMBIGUOUS)} across 3 runs      │ {consistent_after}/{len(AMBIGUOUS)} across 3 runs (or surfaces rule gap) │
│ Routing contract             │ None declared         │ Skip patch + dev-only            │
│ Output contract              │ None                  │ Enum: breaking/security/minor/unknown │
└──────────────────────────────┴───────────────────────┴──────────────────────────────────┘
""")

### Maturity re-assessment

Without these changes, `dependency-update-risk-assessor` is a Level 0 skill — a prompt with no routing contract, no output schema, no regression test.

With them, it moves toward **Level 2 — Verified Skill**:
- **Routing contract** — declared filter: skip patch and dev-only updates
- **Output schema** — enum-constrained: one of `{breaking, security, minor, unknown}`
- **Regression harness** — ambiguous cases tested across 3 runs; inconsistency surfaces decision rules that need to be written
- **CI-embeddable** — output is now machine-readable without string parsing heuristics

What's missing for Level 3: a human approval gate for `breaking` labels before merge is allowed, continuous evaluation as Dependabot batches accumulate real-world data, and a signed release with a changelog of label-definition changes.

---

## ✏️ Explore Further

Replace the dependency list below with updates from your own project. Run all three gaps and observe:
- How many of your Dependabot updates would be filtered by the routing rule?
- Which of your updates produce inconsistent labels across 3 runs — those are your ambiguous cases
- Whether the enum constraint is enough or whether your pipeline needs a more granular schema

In [ ]:
# ── Try it yourself ──────────────────────────────────────────────────────────
# Replace the list below with your own dependency updates.
# bump_type: "major" | "minor" | "patch"
# scope: "prod" | "dev"
# ─────────────────────────────────────────────────────────────────────────────
MY_DEPS = [
    {
        "name": "fastapi", "from": "0.100.0", "to": "0.110.0",
        "bump_type": "minor", "scope": "prod",
        "notes": "Minor release. Adds support for query parameter aliases and improves OpenAPI generation."
    },
    {
        "name": "cryptography", "from": "41.0.0", "to": "42.0.0",
        "bump_type": "major", "scope": "prod",
        "notes": "Major release. Deprecates several legacy cipher modes. Patches CVE-2023-49083 (NULL pointer dereference in PKCS12)."
    },
]

my_to_assess = [d for d in MY_DEPS if should_assess(d)]
print(f"Relevant updates: {len(my_to_assess)} / {len(MY_DEPS)}")

for dep in my_to_assess:
    result = ask(build_assessment_prompt(dep, enforce_schema=True))
    label, valid = extract_label(result["output"], dep["name"])
    print(f"{dep['name']:15} → '{label}' {'✓' if valid else '✗ (not in enum)'}")

---

## What's next

This notebook showed routing accuracy and output schema enforcement. The companion notebooks in this series cover different dominant dimensions:

- **[profile-bio Before & After](https://colab.research.google.com/github/SriharshaCR/blogs/blob/main/assets/notebooks/how-resourceful-is-your-ai-skill/03-hands-on/01_profile_bio_before_after.ipynb)** — token budget + data handling + behavioral regression
- **[Sprint Changelog Before & After](https://colab.research.google.com/github/SriharshaCR/blogs/blob/main/assets/notebooks/how-resourceful-is-your-ai-skill/03-hands-on/02_sprint_changelog_before_after.ipynb)** — output schema compliance + token scaling

→ **[Read the full series: How Resourceful Is Your AI Skill?](https://sriharshacr.github.io/blogs/how-resourceful-is-your-ai-skill/)**

---

*Part of the [open-skills](https://github.com/SriharshaCR/open-skills) project — AI skills built and shared openly.*